In [1]:
import os
import glob
import re
import rioxarray
import xarray as xr
import numpy as np
import pdb
from pathlib import Path
import matplotlib.pyplot as plt

In [3]:
basepath = Path("D:/MyDrive/Stability/RawData")

region_map = {
        "KZSO": "Kotzebue_Sound",
        "OTKZ": "Outer_Kotzebue",
        "PLWR": "Point_Lay-Wainwright",
        "PHPL": "Pt_Hope-Pt_Lay",
        "ULSP": "Uelen-SewardPenn",
        "RUSS": "Russia",
        "NEKS": "NE_Kotzebue_Sound",
        "RURE": "Russia_redo",
}

# Output directory
outdir = basepath / "Monthly_Averages" / "Chukchi"
outdir.mkdir(parents=True, exist_ok=True)

# Date regex
date_regex = re.compile(r'(\d{4})(\d{2})\d{2}')

# Loop over regions
for code, subdir in region_map.items():
    print(f"Processing region {subdir}")
    
    strain_folder = basepath / subdir / "Strain_Files_2025"
    if not strain_folder.exists():
        print(f"Skipping missing folder: {strain_folder}")
        continue

    # Initialize monthly sums and counts per region
    monthly_sum = {m: None for m in range(1, 13)}
    monthly_count = {m: None for m in range(1, 13)}
    monthly_ref = {m: None for m in range(1, 13)}

    tifs = list(strain_folder.glob("*.tif"))
    print(f"Found {len(tifs)} files in {subdir}")

    for tif in tifs:
        fname = tif.name
        match = date_regex.search(fname)
        if not match:
            print(f"Skipping {tif}, cannot parse date")
            continue

        month = int(match.group(2))

        try:
            ds = rioxarray.open_rasterio(
                tif, masked=True, chunks={'x': 500, 'y': 500}
            ).squeeze()

            valid_mask = ~ds.isnull()

            if monthly_sum[month] is None:
                monthly_sum[month] = ds.fillna(0).copy()
                monthly_count[month] = valid_mask.astype(int).copy()
                monthly_ref[month] = ds
            else:
                ds_aligned = ds.rio.reproject_match(monthly_sum[month])
                valid_mask_aligned = ~ds_aligned.isnull()

                monthly_sum[month].data += ds_aligned.fillna(0).data
                monthly_count[month].data += valid_mask_aligned.astype(int).data

        except Exception as e:
            print(f"Could not read {tif}: {e}")

    # Compute & save monthly means per region
    for month in range(1, 13):
        if monthly_sum[month] is None:
            print(f"No data for month {month} in {subdir}, skipping")
            continue

        mean = monthly_sum[month] / monthly_count[month]
        mean = mean.where(monthly_count[month] > 1)

        mean_file = outdir / f"{subdir}_Monthly_Mean_{month:02d}.tif"
        count_file = outdir / f"{subdir}_Monthly_Count_{month:02d}.tif"

        mean.rio.to_raster(mean_file)
        monthly_count[month].rio.to_raster(count_file)

        print(f"Saved {mean_file} and {count_file}")

Processing region Kotzebue_Sound
Found 133 files in Kotzebue_Sound


C:\Users\aeinhorn\AppData\Local\anaconda3\envs\ksa206_environment\lib\site-packages\dask\_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Kotzebue_Sound_Monthly_Mean_01.tif and D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Kotzebue_Sound_Monthly_Count_01.tif


C:\Users\aeinhorn\AppData\Local\anaconda3\envs\ksa206_environment\lib\site-packages\dask\_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Kotzebue_Sound_Monthly_Mean_02.tif and D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Kotzebue_Sound_Monthly_Count_02.tif


...

Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Russia_redo_Monthly_Mean_10.tif and D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Russia_redo_Monthly_Count_10.tif


C:\Users\aeinhorn\AppData\Local\anaconda3\envs\ksa206_environment\lib\site-packages\dask\_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Russia_redo_Monthly_Mean_11.tif and D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Russia_redo_Monthly_Count_11.tif


C:\Users\aeinhorn\AppData\Local\anaconda3\envs\ksa206_environment\lib\site-packages\dask\_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


Saved D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Russia_redo_Monthly_Mean_12.tif and D:\MyDrive\Stability\RawData\Monthly_Averages\Chukchi\Russia_redo_Monthly_Count_12.tif
